# 03. Donor-level analysis, day 28 vs day 0

Input: `annotated.h5ad` from notebook 02. Replication is at the donor level: 5 donors have both day 0 and day 28
(4 for NK after the cycling cells are split off). Everything here is exploratory, because with 4 to 5 donors an exact
test cannot return a p-value below 0.0625 (0.125 with 4 donors).

1. Pseudobulk per donor x day x cell type
2. Donor-level DE (PyDESeq2) before and after splitting cycling cells
3. Cell-level (naive) test vs donor-level test on the same cells
4. Variance partitioning, donor spread and TF activity
5. Composition changes and the summary figure

In [ ]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

# ---- edit these two paths ----------------------------------------------------
data_dir = Path("data")                        # folder with the extracted GSE217930 files
out_dir = Path("E:/malaria_scrna_results")     # large intermediate files, needs about 5 GB free
# ------------------------------------------------------------------------------
fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
for d in (out_dir, fig_dir, tab_dir):
    d.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = fig_dir
sc.set_figure_params(dpi=100, dpi_save=200, facecolor="white")
print({p: version(p) for p in ['scanpy', 'anndata', 'pandas', 'numpy', 'pydeseq2', 'decoupler']})


## Load

In [ ]:
adata = sc.read_h5ad(out_dir / "annotated.h5ad")
if str(adata.var_names[0]).startswith("ENSG") and "gene_symbol" in adata.var:
    adata.var["ensembl_id"] = adata.var_names
    sym = adata.var["gene_symbol"].astype(str)
    bad = sym.isin(["nan", "", "None"])
    sym[bad] = adata.var_names[bad]
    adata.var_names = sym
    adata.var_names_make_unique()
adata.obs["day"] = adata.obs["day"].astype(str)
CTS = ["B", "CD4 T", "CD8 T", "NK"]
DAYS = ["0", "28"]
print(adata)
print(adata.obs["cell_type"].value_counts().to_dict())


## 1. Pseudobulk
Raw counts are summed per donor, day and cell type; samples with fewer than 20 cells are dropped.

In [ ]:
import scipy.sparse as sp

mask = ~adata.obs["exclude"].values
obs = adata.obs.loc[mask].copy()
counts = sp.csr_matrix(adata.layers["counts"][mask])

obs["day"] = obs["day"].astype(str)
obs["sample"] = obs["donor"].astype(str) + "_" + obs["day"]
key = obs["sample"] + "|" + obs["cell_type"].astype(str)

codes, uniques = pd.factorize(key)
n = len(codes)
M = sp.csr_matrix((np.ones(n), (codes, np.arange(n))), shape=(len(uniques), n))
pb_counts = (M @ counts).astype(np.int32)
n_cells = np.bincount(codes)

# Sample-level metadata
meta_cols = [c for c in ["donor", "day", "group", "batch", "age_group"] if c in obs]
sample_meta = obs.groupby("sample", observed=True)[meta_cols].first()

pb_obs = pd.DataFrame({
    "sample": [u.split("|")[0] for u in uniques],
    "cell_type": [u.split("|")[1] for u in uniques],
    "n_cells": n_cells,
})
pb_obs = pb_obs.join(sample_meta, on="sample")
pb_obs.index = uniques

pb = sc.AnnData(X=pb_counts, obs=pb_obs, var=adata.var[["ensembl_id"]].copy())
print("pseudobulk samples before filtering:", pb.n_obs)

# Keep pseudobulk samples with at least 20 cells
MIN_CELLS = 20
pb = pb[pb.obs["n_cells"] >= MIN_CELLS].copy()
print("after filtering (>= %d cells):" % MIN_CELLS, pb.n_obs)

# How many samples per day, and how many donors have both day 0 and day 28?
print(pb.obs.pivot_table(index="cell_type", columns="day", values="n_cells",
                         aggfunc="count", fill_value=0))

paired = {}
for ct, sub in pb.obs.groupby("cell_type", observed=True):
    d0 = set(sub.loc[sub["day"] == "0", "donor"])
    d28 = set(sub.loc[sub["day"] == "28", "donor"])
    paired[ct] = len(d0 & d28)
print("\ndonors with both day 0 and day 28:")
print(pd.Series(paired).sort_values(ascending=False))

pb.write_h5ad(out_dir / "pseudobulk.h5ad")
pb.obs.to_csv(tab_dir / "pseudobulk_samples.csv")
print("saved pseudobulk.h5ad")


In [ ]:
obs = adata.obs.loc[~adata.obs["exclude"]].copy()
obs["day"] = obs["day"].astype(str)
obs["sample"] = obs["donor"].astype(str) + "_" + obs["day"]

# Cells per donor and day for the sparse cell types (before any cutoff)
for ct in ["Monocyte", "cDC", "pDC"]:
    sub = obs[obs["cell_type"] == ct]
    print(f"\n{ct}: cells per donor and day")
    print(pd.crosstab(sub["donor"], sub["day"]))

# Monocyte subtypes at day 0
mono = obs[obs["cell_type"] == "Monocyte"]
print("\nMonocyte subtypes by day")
print(pd.crosstab(mono["cell_subtype"], mono["day"]))

# Total cells per sample, and each cell type's share of the sample
print("\nTotal cells per sample")
print(obs.groupby("sample").size().sort_values())

share = pd.crosstab(obs["sample"], obs["cell_type"], normalize="index").round(3)
print("\nShare of sample (fraction)")
print(share[["Monocyte", "cDC", "pDC"]])


## 2. Donor-level differential expression (PyDESeq2, `~ donor + day`)

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

pb = sc.read_h5ad(out_dir / "pseudobulk.h5ad")
pb.obs["day"] = pb.obs["day"].astype(str)

res_all, summary = {}, []
for ct in ["B", "CD4 T", "CD8 T", "NK"]:
    sub = pb[(pb.obs["cell_type"] == ct) & pb.obs["day"].isin(["0", "28"])].copy()

    # Keep only donors that have both time points
    n_days = sub.obs.groupby("donor", observed=True)["day"].nunique()
    sub = sub[sub.obs["donor"].isin(n_days[n_days == 2].index)].copy()

    X = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    counts_df = pd.DataFrame(X.astype(int), index=sub.obs_names, columns=sub.var_names)
    counts_df = counts_df.loc[:, counts_df.sum(axis=0) >= 10]   # drop very low-count genes

    meta = sub.obs[["donor", "day"]].astype(str)
    try:
        dds = DeseqDataSet(counts=counts_df, metadata=meta, design="~ donor + day")
    except TypeError:                                   # older PyDESeq2 versions
        dds = DeseqDataSet(counts=counts_df, metadata=meta, design_factors=["donor", "day"])
    dds.deseq2()

    stat = DeseqStats(dds, contrast=["day", "28", "0"])
    stat.summary()
    res = stat.results_df.copy()
    res.to_csv(tab_dir / f"pseudobulk_DE_{ct.replace(' ', '_')}_day28_vs_day0.csv")
    res_all[ct] = res

    summary.append({
        "cell_type": ct,
        "donors": sub.obs["donor"].nunique(),
        "samples": sub.n_obs,
        "genes_tested": int(res["padj"].notna().sum()),
        "DEGs_padj<0.05": int((res["padj"] < 0.05).sum()),
        "DEGs_padj<0.05_|LFC|>1": int(((res["padj"] < 0.05) & (res["log2FoldChange"].abs() > 1)).sum()),
    })
    print(ct, "done")

summary = pd.DataFrame(summary)
summary.to_csv(tab_dir / "pseudobulk_DE_summary.csv", index=False)
print(summary)


Do the donor-level DEGs move the same way in every donor? NK shows changes in cycling genes, which motivates splitting cycling cells off.

In [ ]:
def log_cpm(sub):
    X = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    X = X.astype(float)
    cpm = X / X.sum(axis=1, keepdims=True) * 1e6
    return pd.DataFrame(np.log2(cpm + 1), index=sub.obs_names, columns=sub.var_names)

cyc = ["MKI67", "TOP2A", "PCLAF", "TYMS", "RRM2", "CDK1", "BIRC5", "UBE2C"]
rows, per_donor = [], {}

for ct in ["B", "CD4 T", "CD8 T", "NK"]:
    sub = pb[(pb.obs["cell_type"] == ct) & pb.obs["day"].isin(["0", "28"])].copy()
    nd = sub.obs.groupby("donor", observed=True)["day"].nunique()
    sub = sub[sub.obs["donor"].isin(nd[nd == 2].index)].copy()
    L = log_cpm(sub)

    d = {}
    for dn in sorted(sub.obs["donor"].unique()):
        o = sub.obs
        s28 = o.index[(o["donor"] == dn) & (o["day"] == "28")][0]
        s0 = o.index[(o["donor"] == dn) & (o["day"] == "0")][0]
        d[dn] = L.loc[s28] - L.loc[s0]          # per-donor change, day 28 minus day 0
    D = pd.DataFrame(d)
    per_donor[ct] = D

    r = res_all[ct].dropna(subset=["padj"])
    sig = r[r["padj"] < 0.05].index
    agree = np.sign(D.loc[sig]).eq(np.sign(r.loc[sig, "log2FoldChange"]), axis=0).sum(axis=1)
    rows.append({"cell_type": ct, "DEGs": len(sig), "donors": D.shape[1],
                 "all donors agree": int((agree == D.shape[1]).sum()),
                 "4 donors agree": int((agree == D.shape[1] - 1).sum()),
                 "3 or fewer agree": int((agree <= D.shape[1] - 2).sum())})

print(pd.DataFrame(rows).to_string(index=False))

# NK: cell numbers per sample and cycling-gene change per donor
print("\nNK cells per pseudobulk sample")
nk = pb.obs[(pb.obs["cell_type"] == "NK") & pb.obs["day"].isin(["0", "28"])]
print(nk.pivot_table(index="donor", columns="day", values="n_cells", aggfunc="first"))

print("\nNK: change in cycling genes per donor (log2, day 28 minus day 0)")
print(per_donor["NK"].loc[[g for g in cyc if g in per_donor["NK"].index]].round(2))


## Cycling cells as their own group, then repeat the donor-level DE
Cells with cycling score above 0.5 become group `Cycling`; the four lineage groups are re-run without them.

In [ ]:
import scipy.sparse as sp
from scipy.stats import wilcoxon
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

THR = 0.5
adata.obs["cell_type_lineage"] = adata.obs["cell_type"].astype(str)
adata.obs["cell_type_final"] = adata.obs["cell_type_lineage"]
is_cyc = (adata.obs["cycling_score"] > THR) & ~adata.obs["exclude"]
adata.obs.loc[is_cyc, "cell_type_final"] = "Cycling"
adata.obs["cell_type_final"] = adata.obs["cell_type_final"].astype("category")
print(adata.obs["cell_type_final"].value_counts())

# 1) Cycling fraction per sample, and a paired exact test (day 0 vs day 28)
o = adata.obs.loc[~adata.obs["exclude"]].copy()
o["day"] = o["day"].astype(str)
o["sample"] = o["donor"].astype(str) + "_" + o["day"]
tot = o.groupby("sample").size()
cyc = o[o["cell_type_final"] == "Cycling"].groupby("sample").size().reindex(tot.index, fill_value=0)
share = pd.DataFrame({"cycling": cyc, "total": tot, "pct": (cyc / tot * 100).round(2)})
share[["donor", "day"]] = share.index.to_series().str.split("_", expand=True)
wide = share.pivot(index="donor", columns="day", values="pct")
print("\ncycling % per donor and day\n", wide)
paired = wide[["0", "28"]].dropna()
print("paired Wilcoxon day 0 vs 28: n =", len(paired),
      "| p =", round(wilcoxon(paired["0"], paired["28"]).pvalue, 3))
share.to_csv(tab_dir / "cycling_fraction_per_sample.csv")

# 2) Rebuild pseudobulk with the cycling cells as their own group
counts = sp.csr_matrix(adata.layers["counts"][~adata.obs["exclude"].values])
key = o["sample"] + "|" + o["cell_type_final"].astype(str)
codes, uniques = pd.factorize(key)
M = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                  shape=(len(uniques), len(codes)))
meta_cols = [c for c in ["donor", "day", "group", "batch", "age_group"] if c in o]
sample_meta = o.groupby("sample", observed=True)[meta_cols].first()
pb_obs = pd.DataFrame({"sample": [u.split("|")[0] for u in uniques],
                       "cell_type": [u.split("|")[1] for u in uniques],
                       "n_cells": np.bincount(codes)}).join(sample_meta, on="sample")
pb_obs.index = uniques
pb2 = sc.AnnData(X=(M @ counts).astype(np.int32), obs=pb_obs, var=adata.var[["ensembl_id"]].copy())
pb2 = pb2[pb2.obs["n_cells"] >= 20].copy()
pb2.write_h5ad(out_dir / "pseudobulk_cyclingsplit.h5ad")
print("\npseudobulk samples:", pb2.n_obs)

# 3) Donor-level DE on the four lineage groups, cycling cells removed
def run_de(pb_, ct):
    sub = pb_[(pb_.obs["cell_type"] == ct) & pb_.obs["day"].isin(["0", "28"])].copy()
    nd = sub.obs.groupby("donor", observed=True)["day"].nunique()
    sub = sub[sub.obs["donor"].isin(nd[nd == 2].index)].copy()
    X = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    cdf = pd.DataFrame(X.astype(int), index=sub.obs_names, columns=sub.var_names)
    cdf = cdf.loc[:, cdf.sum(axis=0) >= 10]
    meta = sub.obs[["donor", "day"]].astype(str)
    try:
        dds = DeseqDataSet(counts=cdf, metadata=meta, design="~ donor + day")
    except TypeError:
        dds = DeseqDataSet(counts=cdf, metadata=meta, design_factors=["donor", "day"])
    dds.deseq2()
    st = DeseqStats(dds, contrast=["day", "28", "0"])
    st.summary()
    return st.results_df.copy(), sub.obs["donor"].nunique(), sub.n_obs

rows2, res2 = [], {}
for ct in ["B", "CD4 T", "CD8 T", "NK"]:
    r, nd_, ns_ = run_de(pb2, ct)
    r.to_csv(tab_dir / f"pseudobulk_DE_{ct.replace(' ', '_')}_day28_vs_day0_cyclingsplit.csv")
    res2[ct] = r
    rows2.append({"cell_type": ct, "donors": nd_, "samples": ns_,
                  "DEGs_after": int((r["padj"] < 0.05).sum()),
                  "DEGs_after_|LFC|>1": int(((r["padj"] < 0.05) & (r["log2FoldChange"].abs() > 1)).sum())})

old = pd.read_csv(tab_dir / "pseudobulk_DE_summary.csv")[["cell_type", "donors", "DEGs_padj<0.05"]]
old.columns = ["cell_type", "donors_before", "DEGs_before"]
cmp = pd.DataFrame(rows2).merge(old, on="cell_type")
cmp.to_csv(tab_dir / "pseudobulk_DE_before_vs_after_cyclingsplit.csv", index=False)
print("\n", cmp)


## 3. Cell-level (naive) test vs donor-level test on the same cells and donors

In [ ]:
# STEP 1: naive cell-level test vs donor-level test, same cells and donors
cts = ["B", "CD4 T", "CD8 T", "NK"]

def bh(p):
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=1.0)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(ranked, 1.0)
    return out

def frac_expr(X):
    return np.asarray((X > 0).mean(axis=0)).ravel()

rows, naive = [], {}
for ct in cts:
    sp_ = pb2[(pb2.obs["cell_type"] == ct) & pb2.obs["day"].isin(["0", "28"])]
    nd = sp_.obs.groupby("donor", observed=True)["day"].nunique()
    donors = list(nd[nd == 2].index)

    keep = ((adata.obs["cell_type_final"] == ct)
            & adata.obs["donor"].isin(donors)
            & adata.obs["day"].astype(str).isin(["0", "28"])).values
    v = adata[keep]
    day = v.obs["day"].astype(str).values
    tmp = sc.AnnData(X=v.layers["lognorm"].copy(),
                     obs=pd.DataFrame({"day": pd.Categorical(day, categories=["0", "28"])},
                                      index=v.obs_names.astype(str)),
                     var=pd.DataFrame(index=v.var_names))
    sc.pp.filter_genes(tmp, min_cells=10)
    sc.tl.rank_genes_groups(tmp, "day", groups=["28"], reference="0", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(tmp, group="28")

    m28 = (tmp.obs["day"] == "28").values
    pct28 = pd.Series(frac_expr(tmp.X[m28]), index=tmp.var_names)
    pct0 = pd.Series(frac_expr(tmp.X[~m28]), index=tmp.var_names)
    df["pct_day28"] = df["names"].map(pct28).values
    df["pct_day0"] = df["names"].map(pct0).values
    pv = df["pvals"].fillna(1.0).values
    df["padj_BH"] = bh(pv)
    df["padj_Bonf"] = np.minimum(pv * len(df), 1.0)
    df["padj_Bonf_allgenes"] = np.minimum(pv * adata.n_vars, 1.0)
    df.to_csv(tab_dir / f"naive_wilcoxon_{ct.replace(' ', '_')}_day28_vs_day0.csv", index=False)
    naive[ct] = df

    seurat_like = df[(df["padj_Bonf_allgenes"] < 0.05)
                     & (df["logfoldchanges"].abs() > 0.25)
                     & ((df["pct_day28"] >= 0.1) | (df["pct_day0"] >= 0.1))]
    naive_bh = df[df["padj_BH"] < 0.05]

    r = res2[ct]
    n_tested = int(r["pvalue"].notna().sum())
    donor_bh = r.index[r["padj"] < 0.05]
    donor_bonf = r.index[(r["pvalue"] * n_tested) < 0.05]

    rows.append({
        "cell_type": ct, "donors": len(donors),
        "cells_day0": int((~m28).sum()), "cells_day28": int(m28.sum()),
        "naive_BH": len(naive_bh),
        "naive_Bonferroni": int((df["padj_Bonf"] < 0.05).sum()),
        "naive_Seurat_like": len(seurat_like),
        "donor_BH": len(donor_bh),
        "donor_Bonferroni": len(donor_bonf),
        "overlap_naiveBH_donorBH": len(set(naive_bh["names"]) & set(donor_bh)),
    })
    print(ct, "| donors used:", donors)

calib = pd.DataFrame(rows)
calib.to_csv(tab_dir / "calibration_naive_vs_donor_level.csv", index=False)
print("\n", calib.to_string(index=False))


Are the genes found only by the naive test consistent across donors? (Same direction in every donor, against chance.)

In [ ]:
# STEP 1b: are the genes found only by the naive test consistent across donors?
cts = ["B", "CD4 T", "CD8 T", "NK"]
DAYS = ["0", "28"]

def paired_donors(pb_, ct):
    sub = pb_[(pb_.obs["cell_type"] == ct) & pb_.obs["day"].isin(DAYS)]
    nd = sub.obs.groupby("donor", observed=True)["day"].nunique()
    return sorted(nd[nd == 2].index)

rows = []
for ct in cts:
    donors = paired_donors(pb2, ct)
    D = len(donors)
    sub = pb2[(pb2.obs["cell_type"] == ct) & pb2.obs["donor"].isin(donors)
              & pb2.obs["day"].isin(DAYS)].copy()
    C = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    C = C.astype(float)
    L = np.log2(C / C.sum(axis=1, keepdims=True) * 1e6 + 1)
    don = sub.obs["donor"].astype(str).values
    day = sub.obs["day"].astype(str).values
    change = np.column_stack([L[(don == d) & (day == "28")][0] - L[(don == d) & (day == "0")][0]
                              for d in donors])
    genes = list(sub.var_names)
    chg = pd.DataFrame(change, index=genes, columns=donors)
    expressed = pd.Series(C.mean(axis=0) >= 20, index=genes)       # reliable per-donor change

    nv = naive[ct]
    nsig = nv[(nv["padj_BH"] < 0.05) & nv["names"].isin(genes)]
    dsig = set(res2[ct].index[res2[ct]["padj"] < 0.05])
    groups = {"naive_only": nsig[~nsig["names"].isin(dsig)],
              "found_by_both": nsig[nsig["names"].isin(dsig)]}
    for label, g in groups.items():
        g = g[g["names"].map(expressed).fillna(False).astype(bool)]
        if len(g) == 0:
            continue
        sgn = np.sign(g["logfoldchanges"].values)[:, None]
        agree = (np.sign(chg.loc[g["names"]].values) == sgn).sum(axis=1)
        rows.append({"cell_type": ct, "group": label, "donors": D, "genes": len(g),
                     "pct_all_donors_agree": round(100 * (agree == D).mean(), 1),
                     "pct_all_but_one_agree": round(100 * (agree >= D - 1).mean(), 1),
                     "chance_all": round(100 * 0.5 ** D, 1),
                     "chance_all_but_one": round(100 * (1 + D) * 0.5 ** D, 1)})
out = pd.DataFrame(rows)
out.to_csv(tab_dir / "naive_only_DEG_donor_consistency.csv", index=False)
print("Share of DEGs whose direction matches in every donor (expressed genes only)")
print(out.to_string(index=False))


## 4a. How much expression variance is donor and how much is day?

In [ ]:
# STEP 2: between-donor variability (paired donors, day 0 vs day 28)
cts = ["B", "CD4 T", "CD8 T", "NK"]
DAYS = ["0", "28"]

def paired_donors(pb_, ct):
    sub = pb_[(pb_.obs["cell_type"] == ct) & pb_.obs["day"].isin(DAYS)]
    nd = sub.obs.groupby("donor", observed=True)["day"].nunique()
    return sorted(nd[nd == 2].index)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

def log_cpm(C):                        # samples x genes raw counts -> log2 CPM
    return np.log2(C / C.sum(axis=1, keepdims=True) * 1e6 + 1)

# ---- 2a) How much of the variance is donor, how much is day? (full pseudobulk) ----
vp_rows, vp_tables = [], {}
for ct in cts:
    donors = paired_donors(pb2, ct)
    D = len(donors)
    sub = pb2[(pb2.obs["cell_type"] == ct) & pb2.obs["donor"].isin(donors)
              & pb2.obs["day"].isin(DAYS)].copy()
    C = to_dense(sub.X).astype(float)
    don = sub.obs["donor"].astype(str).values
    day = sub.obs["day"].astype(str).values
    good = C.mean(axis=0) >= 20                       # well-expressed genes only
    L = log_cpm(C)[:, good]
    gm = L.mean(axis=0)
    ss_tot = ((L - gm) ** 2).sum(axis=0)
    ss_don = sum((don == d).sum() * (L[don == d].mean(axis=0) - gm) ** 2 for d in donors)
    ss_day = sum((day == dd).sum() * (L[day == dd].mean(axis=0) - gm) ** 2 for dd in DAYS)
    ss_res = ss_tot - ss_don - ss_day
    ok = ss_tot > 0
    tab = pd.DataFrame({"donor": ss_don / ss_tot, "day": ss_day / ss_tot,
                        "residual": ss_res / ss_tot},
                       index=sub.var_names[good])[ok]
    vp_tables[ct] = tab
    df_tot = 2 * D - 1
    vp_rows.append({"cell_type": ct, "donors": D, "genes": len(tab),
                    "donor_median": tab["donor"].median(),
                    "day_median": tab["day"].median(),
                    "residual_median": tab["residual"].median(),
                    "donor_null": (D - 1) / df_tot, "day_null": 1 / df_tot,
                    "residual_null": (D - 1) / df_tot,
                    "pct_genes_donor>day": float((tab["donor"] > tab["day"]).mean() * 100)})
    tab.to_csv(tab_dir / f"variance_partition_{ct.replace(' ', '_')}.csv")
vp = pd.DataFrame(vp_rows).round(3)
vp.to_csv(tab_dir / "variance_partition_summary.csv", index=False)
print("Share of log-CPM variance per gene (median over genes), against the share expected by chance")
print(vp.to_string(index=False))


Observed day share against a label-flip null (day labels flipped within donors), and the day 0 / day 28 donor variance ratio with equal cells per sample.

In [ ]:
# STEP 2c: empirical nulls for Step 2 (day labels flipped within donors)
import itertools
from scipy.stats import f as f_dist
cts = ["B", "CD4 T", "CD8 T", "NK"]
DAYS = ["0", "28"]

def paired_donors(pb_, ct):
    sub = pb_[(pb_.obs["cell_type"] == ct) & pb_.obs["day"].isin(DAYS)]
    nd = sub.obs.groupby("donor", observed=True)["day"].nunique()
    return sorted(nd[nd == 2].index)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

def log_cpm(C):
    return np.log2(C / C.sum(axis=1, keepdims=True) * 1e6 + 1)

# ---- (i) variance shares: observed against the flip null, plus F-type ratios ----
rows = []
for ct in cts:
    donors = paired_donors(pb2, ct)
    D = len(donors)
    sub = pb2[(pb2.obs["cell_type"] == ct) & pb2.obs["donor"].isin(donors)
              & pb2.obs["day"].isin(DAYS)].copy()
    C = to_dense(sub.X).astype(float)
    don = sub.obs["donor"].astype(str).values
    day = sub.obs["day"].astype(str).values
    L = log_cpm(C)[:, C.mean(axis=0) >= 20]
    A = np.column_stack([L[(don == d) & (day == "0")][0] for d in donors])    # genes x donors
    B = np.column_stack([L[(don == d) & (day == "28")][0] for d in donors])

    def stats(flip):
        X0 = np.where(flip[None, :], B, A)
        X1 = np.where(flip[None, :], A, B)
        m0, m1 = X0.mean(axis=1), X1.mean(axis=1)
        gm = (m0 + m1) / 2
        dm = (X0 + X1) / 2
        ss_tot = ((X0 - gm[:, None]) ** 2).sum(axis=1) + ((X1 - gm[:, None]) ** 2).sum(axis=1)
        ss_don = 2 * ((dm - gm[:, None]) ** 2).sum(axis=1)
        ss_day = D * ((m0 - gm) ** 2 + (m1 - gm) ** 2)
        ss_res = ss_tot - ss_don - ss_day
        return ss_tot, ss_don, ss_day, ss_res

    ss_tot, ss_don, ss_day, ss_res = stats(np.zeros(D, dtype=bool))
    ok = (ss_tot > 0) & (ss_res > 0)
    obs_day = np.median((ss_day / ss_tot)[ok])
    null_day = []
    for flip in itertools.product([False, True], repeat=D):
        flip = np.array(flip)
        if not flip.any() or flip.all():
            continue                                   # true labels and their mirror image
        _, _, sd, _ = stats(flip)
        null_day.append(np.median((sd / ss_tot)[ok]))
    null_day = np.array(null_day)
    rows.append({"cell_type": ct, "donors": D, "genes": int(ok.sum()),
                 "donor_share": np.median((ss_don / ss_tot)[ok]),
                 "day_share": obs_day,
                 "day_share_null_mean": null_day.mean(),
                 "day_share_null_max": null_day.max(),
                 "p_day_flip": (2 + (null_day >= obs_day).sum()) / 2 ** D,
                 "F_donor_median": np.median((ss_don / ss_res)[ok]),
                 "F_donor_null_median": f_dist.median(D - 1, D - 1),
                 "F_day_median": np.median((ss_day / (ss_res / (D - 1)))[ok]),
                 "F_day_null_median": f_dist.median(1, D - 1)})
t1 = pd.DataFrame(rows).round(3)
t1.to_csv(tab_dir / "variance_partition_empirical_null.csv", index=False)
print("Variance shares against the label-flip null (smallest possible p with these donors is 2/2^D)")
print(t1.to_string(index=False))

# ---- (ii) day 0 / day 28 donor variance ratio with a flip p-value (equal cells per sample) ----
R, TOPG = 30, 2000
rng = np.random.default_rng(1)
counts_all = adata.layers["counts"]
day_all = adata.obs["day"].astype(str).values
ct_all = adata.obs["cell_type_final"].astype(str).values
don_all = adata.obs["donor"].astype(str).values
excl = adata.obs["exclude"].values.astype(bool)

rows = []
for ct in cts:
    donors = paired_donors(pb2, ct)
    D = len(donors)
    idx0 = [np.where((ct_all == ct) & (don_all == d) & (day_all == "0") & ~excl)[0] for d in donors]
    idx28 = [np.where((ct_all == ct) & (don_all == d) & (day_all == "28") & ~excl)[0] for d in donors]
    n_min = min(min(len(i) for i in idx0), min(len(i) for i in idx28))
    if n_min < 20:
        print(ct, ": skipped, smallest sample has", n_min, "cells")
        continue
    sub = pb2[(pb2.obs["cell_type"] == ct) & pb2.obs["donor"].isin(donors)
              & pb2.obs["day"].isin(DAYS)]
    top = np.argsort(-to_dense(sub.X).astype(float).sum(axis=0))[:TOPG]
    flips = [np.array(fl) for fl in itertools.product([False, True], repeat=D)]
    obs_ratio, p_reps = [], []
    for _ in range(R):
        M0 = np.vstack([np.asarray(counts_all[rng.choice(i, n_min, replace=False)].sum(axis=0)).ravel()
                        for i in idx0]).astype(float)
        M1 = np.vstack([np.asarray(counts_all[rng.choice(i, n_min, replace=False)].sum(axis=0)).ravel()
                        for i in idx28]).astype(float)
        L0 = log_cpm(M0)[:, top]
        L1 = log_cpm(M1)[:, top]
        lr = []
        for fl in flips:
            X0 = np.where(fl[:, None], L1, L0)
            X1 = np.where(fl[:, None], L0, L1)
            lr.append(np.log(np.median(X0.var(axis=0, ddof=1)) / np.median(X1.var(axis=0, ddof=1))))
        lr = np.array(lr)
        obs_ratio.append(np.exp(lr[0]))                 # first flip pattern = no flips
        p_reps.append((np.abs(lr) >= abs(lr[0]) - 1e-12).mean())
    rows.append({"cell_type": ct, "donors": D, "cells_per_sample": n_min,
                 "ratio_matched": np.mean(obs_ratio), "p_flip": np.mean(p_reps)})
t2 = pd.DataFrame(rows).round(3)
t2.to_csv(tab_dir / "donor_variance_ratio_flip_test.csv", index=False)
print("\nDonor variance ratio day 0 / day 28 with flip test (smallest possible p is 2/2^D)")
print(t2.to_string(index=False))


## 4b. Transcription-factor activity from the donor-level Wald statistics (CollecTRI, decoupler ULM)
p-values come from a regression across genes, not across donors, so treat them as hypothesis generating.

In [ ]:
# STEP 3: transcription factor activity from the donor-level DE statistics
import decoupler as dc

def bh(p):
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=1.0)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(ranked, 1.0)
    return out

# 3a) TF-target network (CollecTRI); needs internet the first time
try:
    if hasattr(dc, "op"):                                   # decoupler 2.x
        net = dc.op.collectri(organism="human")
    else:                                                   # decoupler 1.x
        net = dc.get_collectri(organism="human", split_complexes=False)
except Exception as e:
    raise RuntimeError("Could not download the CollecTRI network. Check the internet "
                       "connection and rerun this cell. Original error: " + repr(e))
net = net[["source", "target", "weight"]].drop_duplicates(["source", "target"])
print("network:", net["source"].nunique(), "TFs,", len(net), "edges")

# 3b) Input: one row per cell type, one column per gene (Wald statistic, day 28 vs day 0)
cts = [ct for ct in ["B", "CD4 T", "CD8 T", "NK"] if ct in res2]
mat = pd.DataFrame({ct: res2[ct]["stat"] for ct in cts}).T
mat = mat.loc[:, ~mat.columns.duplicated()].fillna(0.0)
print("input matrix:", mat.shape)

# 3c) Univariate linear model per TF
if hasattr(dc, "mt"):
    est, pv = dc.mt.ulm(mat, net, tmin=5)
else:
    est, pv = dc.run_ulm(mat=mat, net=net, source="source", target="target",
                         weight="weight", min_n=5, verbose=False)
est = pd.DataFrame(est); pv = pd.DataFrame(pv)
padj = pv.copy()
for ct in padj.index:
    padj.loc[ct] = bh(pv.loc[ct].values)

est.to_csv(tab_dir / "TF_activity_score_day28_vs_day0.csv")
pv.to_csv(tab_dir / "TF_activity_pvalue_day28_vs_day0.csv")
padj.to_csv(tab_dir / "TF_activity_padjBH_day28_vs_day0.csv")

# 3d) Top TFs per cell type (positive = more active at day 28, negative = more active at day 0)
print("\nTop TFs per cell type")
for ct in est.index:
    s = est.loc[ct].sort_values()
    t = pd.DataFrame({"score": s, "p": pv.loc[ct, s.index], "padj": padj.loc[ct, s.index]})
    print(f"\n{ct}: more active at day 0")
    print(t.head(8).round(3).to_string())
    print(f"{ct}: more active at day 28")
    print(t.tail(8).iloc[::-1].round(3).to_string())

# 3e) The transcription factors named in the paper (HOMER motif results)
paper_tfs = ["NFKB1", "NFKB2", "RELA", "RELB", "REL", "IRF1", "IRF2", "IRF3", "IRF4",
             "IRF7", "IRF8", "IRF9", "STAT1", "STAT2", "STAT3", "STAT5A", "STAT5B",
             "JUN", "JUNB", "JUND", "FOS", "FOSB", "ATF3", "BATF", "TP53"]
present = [t for t in paper_tfs if t in est.columns]
missing = [t for t in paper_tfs if t not in est.columns]
tab = pd.concat({"score": est[present].T, "p": pv[present].T}, axis=1).round(3)
tab.to_csv(tab_dir / "TF_activity_paper_TFs.csv")
print("\nTFs named in the paper (score and p-value per cell type)")
print(tab.to_string())
print("\nnot scored (too few targets in the network or data):", missing)


## 5. Composition
Log-proportion change per donor, exact sign-flip test, donors with at least 5 cells of that type in both samples.

In [ ]:
# Composition per sample (doublets excluded; cycling cells are their own group)
o = adata.obs.loc[~adata.obs["exclude"]].copy()
o["sample"] = o["donor"].astype(str) + "_" + o["day"]
counts = pd.crosstab(o["sample"], o["cell_type_final"].astype(str))
prop = counts.div(counts.sum(axis=1), axis=0)

meta = o.groupby("sample", observed=True)[["donor", "day"]].first()
donors = sorted(d for d in meta["donor"].unique()
                if {"0", "28"} <= set(meta.loc[meta["donor"] == d, "day"]))
print("paired donors:", donors)

import itertools

K = counts.shape[1]
tot = counts.sum(axis=1)
lp = np.log((counts + 0.5).div(tot + 0.5 * K, axis=0))   # log proportion

MIN_N = 5
rows = []
for ct in lp.columns:
    ok = [d for d in donors
          if counts.loc[f"{d}_0", ct] >= MIN_N and counts.loc[f"{d}_28", ct] >= MIN_N]
    base = {"cell_type": ct, "total_cells": int(counts[ct].sum()), "donors_used": len(ok)}
    if len(ok) < 4:
        rows.append({**base, "note": "not tested (<4 donors with >=5 cells in both samples)"})
        continue
    diff = np.array([lp.loc[f"{d}_28", ct] - lp.loc[f"{d}_0", ct] for d in ok])
    fl = np.array(list(itertools.product([1, -1], repeat=len(ok))))
    null = (fl * diff).mean(axis=1)
    p = float((np.abs(null) >= abs(diff.mean()) - 1e-12).mean())
    rows.append({**base,
        "mean_pct_day0":  round(100 * np.mean([prop.loc[f"{d}_0", ct]  for d in ok]), 2),
        "mean_pct_day28": round(100 * np.mean([prop.loc[f"{d}_28", ct] for d in ok]), 2),
        "mean_logprop_change": round(diff.mean(), 3),
        "donors_up": int((diff > 0).sum()), "donors_down": int((diff < 0).sum()),
        "p_flip": round(p, 3), "min_possible_p": round(2 / 2 ** len(ok), 3)})

comp = pd.DataFrame(rows)
m = comp["p_flip"].notna()
comp.loc[m, "padj_BH"] = bh(comp.loc[m, "p_flip"]).round(3)
comp.to_csv(tab_dir / "composition_day28_vs_day0_minN5.csv", index=False)
print(comp.sort_values("p_flip").to_string(index=False))

# paired plot: % of sample per donor, day 0 -> day 28
cts_plot = [c for c in comp.sort_values("p_flip", na_position="last")["cell_type"] if c != "Platelet"]
fig, axes = plt.subplots(2, (len(cts_plot) + 1) // 2, figsize=(14, 6))
for ax, ct in zip(axes.ravel(), cts_plot):
    for d in donors:
        ax.plot([0, 1], [100 * prop.loc[f"{d}_0", ct], 100 * prop.loc[f"{d}_28", ct]],
                marker="o", lw=1)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["day 0", "day 28"])
    ax.set_title(ct, fontsize=9); ax.set_ylabel("% of sample")
for ax in axes.ravel()[len(cts_plot):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(fig_dir / "composition_paired.png", dpi=200, bbox_inches="tight")
plt.show()


## Summary figure

In [ ]:
# reload the saved result tables used in the figure
calib = pd.read_csv(tab_dir / "calibration_naive_vs_donor_level.csv")
consist = pd.read_csv(tab_dir / "naive_only_DEG_donor_consistency.csv")
vp_sum = pd.read_csv(tab_dir / "variance_partition_summary.csv")
vp_null = pd.read_csv(tab_dir / "variance_partition_empirical_null.csv")

CTS = ["B", "CD4 T", "CD8 T", "NK"]
x = np.arange(len(CTS)); w = 0.38

comp = pd.read_csv(tab_dir / "composition_day28_vs_day0_minN5.csv")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# A) cell-level vs donor-level DEG counts (BH < 0.05)
ax = axes[0, 0]
c = calib.set_index("cell_type").loc[CTS]
ax.bar(x - w/2, c["naive_BH"], w, label="cell-level (naive)")
ax.bar(x + w/2, c["donor_BH"], w, label="donor-level (pseudobulk)")
for i, (a, b) in enumerate(zip(c["naive_BH"], c["donor_BH"])):
    ax.text(i - w/2, a, int(a), ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, b, int(b), ha="center", va="bottom", fontsize=8)
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(CTS)
ax.set_ylabel("DEGs, BH < 0.05 (log scale)"); ax.legend(fontsize=8)
ax.set_title("A. Same cells, two tests")

# B) do DEGs go the same way in every donor?
ax = axes[0, 1]
piv = consist.pivot(index="cell_type", columns="group",
                    values="pct_all_donors_agree").reindex(CTS)
g = consist.pivot(index="cell_type", columns="group", values="genes").reindex(CTS)
chance = consist.groupby("cell_type")["chance_all"].first().reindex(CTS)
ax.bar(x - w/2, piv["naive_only"], w, label="found only by naive test")
ax.bar(x + w/2, piv["found_by_both"], w, label="found by both")
ax.scatter(x, chance, marker="_", s=500, c="k", label="expected by chance")
for i, ct in enumerate(CTS):
    for off, grp in [(-w/2, "naive_only"), (w/2, "found_by_both")]:
        if pd.notna(g.loc[ct, grp]):
            ax.text(i + off, piv.loc[ct, grp] + 1, f"n={int(g.loc[ct, grp])}",
                    ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(CTS)
ax.set_ylabel("% of DEGs with same direction in all donors"); ax.legend(fontsize=8)
ax.set_title("B. Donor consistency of DEGs")

# C) donor vs day variance, with the label-flip null for day
ax = axes[1, 0]
v = vp_sum.set_index("cell_type").loc[CTS]
n = vp_null.set_index("cell_type").loc[CTS]
for k, name in enumerate(["donor", "day", "residual"]):
    ax.bar(x + (k - 1) * 0.27, v[f"{name}_median"], 0.27, label=name)
ax.scatter(x, n["day_share_null_mean"], marker="_", s=200, c="k", label="day: label-flip null mean")
ax.scatter(x, n["day_share_null_max"], marker="x", s=25, c="k", label="day: label-flip null max")
ax.set_xticks(x); ax.set_xticklabels(CTS)
ax.set_ylabel("median share of variance per gene")
ax.set_title("C. Donor vs day variance")
ax.set_ylim(0, 0.72)
ax.legend(fontsize=8, ncol=2, loc="upper center")

# D) composition change, day 28 vs day 0
ax = axes[1, 1]
t = comp[comp["p_flip"].notna()].sort_values("mean_logprop_change")
ax.barh(t["cell_type"], t["mean_logprop_change"])
for i, (_, r) in enumerate(t.iterrows()):
    val = r["mean_logprop_change"]
    lab = f"up {int(r['donors_up'])} / down {int(r['donors_down'])}"
    ax.text(val, i, ("  " + lab) if val > 0 else (lab + "  "),
            ha="left" if val > 0 else "right", va="center", fontsize=8)
ax.axvline(0, c="k", lw=0.8)
ax.set_xlim(-2.2, 1.7)
ax.set_xlabel("mean change in log proportion (day 28 minus day 0)")
ax.set_title("D. Composition (exploratory, n = 4-5 donors)")

fig.tight_layout()
fig.savefig(fig_dir / "summary_panel.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved:", fig_dir / "summary_panel.png")
